# QUY TRÌNH TIỀN XỬ LÝ VÀ LÀM SẠCH DỮ LIỆU (HOTEL BOOKING DATA CLEANING)

Notebook này triển khai quy trình làm sạch dữ liệu (Data Cleaning & Preprocessing) bài bản cho tập dữ liệu `hotel_bookings.csv` dựa trên các phát hiện từ bước `understanding.ipynb`.

>**Lưu ý theo yêu cầu**: Notebook này thực hiện làm sạch dữ liệu và lưu giữ kết quả trong biến bộ nhớ `df_clean`, **chưa xuất/ghi ra file CSV mới**.

---

## Mục tiêu quy trình làm sạch
1. **Xử lý dữ liệu trùng lặp (Duplicates Handling)**
2. **Xử lý giá trị khuyết thiếu (Missing Values Handling)**
3. **Xử lý dữ liệu bất thường & Ngoại lai (Anomalies & Outliers)**
4. **Chuẩn hóa kiểu dữ liệu (Data Type Casting & Conversion)**
5. **Kỹ thuật tạo đặc trưng mới (Feature Engineering)**
6. **Kiểm tra và So sánh dữ liệu Trước vs Sau khi làm sạch**

## 1. Khởi tạo môi trường & Đọc dữ liệu ban đầu

In [1]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

df = pd.read_csv('Data/hotel_bookings.csv')
print(f"Đã nạp dữ liệu gốc: {df.shape[0]:,} dòng, {df.shape[1]} cột")

df_clean = df.copy()

Đã nạp dữ liệu gốc: 119,390 dòng, 32 cột


## 2. Bước 1: Xử lý dữ liệu trùng lặp (Handling Duplicates)

Trong tập dữ liệu đặt phòng khách sạn, các đơn trùng lặp 100% tất cả các trường thường xuất phát từ việc ghi nhận trùng (redundant logging). Chúng ta sẽ tiến hành loại bỏ các bản ghi trùng lặp này.

In [2]:
initial_duplicates = df_clean.duplicated().sum()
print(f"Số lượng bản ghi trùng lặp: {initial_duplicates:,} dòng ({initial_duplicates/len(df_clean)*100:.2f}%)")

df_clean.drop_duplicates(inplace=True)

print(f"Kích thước DataFrame sau khi xóa trùng lặp: {df_clean.shape[0]:,} dòng, {df_clean.shape[1]} cột")

Số lượng bản ghi trùng lặp: 31,994 dòng (26.80%)
Kích thước DataFrame sau khi xóa trùng lặp: 87,396 dòng, 32 cột


## 3. Bước 2: Xử lý giá trị khuyết thiếu (Handling Missing Values)

Nhắc lại các cột có missing values:
- `company`: Khách đặt cá nhân -> Điền `0` (thay vì drop cột, để giữ thông tin có qua công ty hay không).
- `agent`: Khách tự đặt không qua đại lý -> Điền `0`.
- `country`: Điền giá trị xuất hiện nhiều nhất (Mode) hoặc `'Unknown'`.
- `children`: Điền `0` cho 4 dòng bị thiếu.

In [3]:
print("Missing values trước khi xử lý:")
print(df_clean.isnull().sum()[df_clean.isnull().sum() > 0])

Missing values trước khi xử lý:
children        4
country       452
agent       12193
company     82137
dtype: int64


In [4]:
df_clean['company'].fillna(0, inplace=True)
df_clean['agent'].fillna(0, inplace=True)

df_clean['country'].fillna('Unknown', inplace=True)

df_clean['children'].fillna(0, inplace=True)

remaining_nulls = df_clean.isnull().sum().sum()
print(f"Tổng số giá trị khuyết thiếu sau khi xử lý: {remaining_nulls}")

Tổng số giá trị khuyết thiếu sau khi xử lý: 0


## 4. Bước 3: Xử lý dữ liệu bất thường & Ngoại lai (Anomalies & Inconsistencies)

### 4.1 Xóa các bản ghi không có khách nào (0 Người lớn, 0 Trẻ em, 0 Trẻ sơ sinh)

In [5]:
zero_guests_filter = (df_clean['adults'] == 0) & (df_clean['children'] == 0) & (df_clean['babies'] == 0)
print(f"Số lượng đơn đặt phòng có 0 khách: {zero_guests_filter.sum()} dòng")

df_clean = df_clean[~zero_guests_filter]
print(f"Kích thước sau khi loại bỏ 0 khách: {df_clean.shape[0]:,} dòng")

Số lượng đơn đặt phòng có 0 khách: 166 dòng
Kích thước sau khi loại bỏ 0 khách: 87,230 dòng


### 4.2 Chuẩn hóa giá trị `'Undefined'` trong các cột phân loại
Theo tài liệu nghiệp vụ của tập dữ liệu:
- Trong cột `meal`: `'Undefined'` tương đương với `'SC'` (Self Catering - Không kèm bữa ăn).
- Trong `market_segment` và `distribution_channel`: Các dòng `'Undefined'` là lỗi nhập liệu, có thể lọc bỏ hoặc thay thế.

In [6]:
df_clean['meal'] = df_clean['meal'].replace('Undefined', 'SC')
print("Phân phối giá trị cột meal sau khi chuẩn hóa:")
print(df_clean['meal'].value_counts())

df_clean = df_clean[df_clean['market_segment'] != 'Undefined']
df_clean = df_clean[df_clean['distribution_channel'] != 'Undefined']
print(f"\nKích thước sau khi chuẩn hóa phân loại: {df_clean.shape[0]:,} dòng")

Phân phối giá trị cột meal sau khi chuẩn hóa:
meal
BB    67907
SC     9883
HB     9080
FB      360
Name: count, dtype: int64

Kích thước sau khi chuẩn hóa phân loại: 87,225 dòng


### 4.3 Xử lý ngoại lai giá phòng (`adr`)
- Loại bỏ các giá trị `adr < 0` (giá phòng âm là dữ liệu lỗi).
- Loại bỏ giá trị `adr == 5400` (giá phòng cực đoan bất thường so với mức trung bình ~100€).

In [7]:
print("Thống kê cột ADR:")
print(df_clean['adr'].describe())

df_clean = df_clean[(df_clean['adr'] >= 0) & (df_clean['adr'] < 5000)]
print(f"\nKích thước sau khi xử lý ngoại lai ADR: {df_clean.shape[0]:,} dòng")

Thống kê cột ADR:
count   87225.00
mean      106.52
std        54.89
min        -6.38
25%        72.25
50%        98.20
75%       134.10
max      5400.00
Name: adr, dtype: float64

Kích thước sau khi xử lý ngoại lai ADR: 87,223 dòng


## 5. Bước 4: Chuyển đổi và chuẩn hóa kiểu dữ liệu (Data Type Casting)

In [8]:
df_clean['children'] = df_clean['children'].astype(int)
df_clean['agent'] = df_clean['agent'].astype(int)
df_clean['company'] = df_clean['company'].astype(int)

df_clean['reservation_status_date'] = pd.to_datetime(df_clean['reservation_status_date'])

month_map = {
'January': 1, 'February': 2, 'March': 3, 'April': 4,
'May': 5, 'June': 6, 'July': 7, 'August': 8,
'September': 9, 'October': 10, 'November': 11, 'December': 12
}

df_clean['arrival_date'] = pd.to_datetime(
df_clean['arrival_date_year'].astype(str) + '-' +
df_clean['arrival_date_month'].map(month_map).astype(str) + '-' +
df_clean['arrival_date_day_of_month'].astype(str)
)

print("Kiểm tra kiểu dữ liệu các cột vừa chuyển đổi:")
print(df_clean[['children', 'agent', 'company', 'reservation_status_date', 'arrival_date']].dtypes)

Kiểm tra kiểu dữ liệu các cột vừa chuyển đổi:
children                            int64
agent                               int64
company                             int64
reservation_status_date    datetime64[ns]
arrival_date               datetime64[ns]
dtype: object


## 6. Bước 5: Kỹ thuật tạo đặc trưng mới (Feature Engineering)

Tạo các thuộc tính phái sinh có giá trị cao cho quá trình phân tích kinh doanh và mô hình hóa:
1. `total_stay`: Tổng số đêm khách lưu trú (`stays_in_weekend_nights` + `stays_in_week_nights`).
2. `total_guests`: Tổng số khách thực tế (`adults` + `children` + `babies`).
3. `is_room_changed`: Cờ đánh dấu khách có bị đổi phòng khác với phòng đã đặt không (`1`: Có đổi, `0`: Đúng phòng).
4. `is_family`: Khách đi theo dạng gia đình có trẻ em / em bé (`1`: Có, `0`: Không).
5. `total_cost`: Tổng số tiền khách phải trả nếu không hủy (`total_stay` * `adr`).

In [9]:
df_clean['total_stay'] = df_clean['stays_in_weekend_nights'] + df_clean['stays_in_week_nights']

df_clean['total_guests'] = df_clean['adults'] + df_clean['children'] + df_clean['babies']

df_clean['is_room_changed'] = (df_clean['reserved_room_type'] != df_clean['assigned_room_type']).astype(int)

df_clean['is_family'] = ((df_clean['children'] > 0) | (df_clean['babies'] > 0)).astype(int)

df_clean['total_cost'] = np.where(df_clean['is_canceled'] == 0, df_clean['total_stay'] * df_clean['adr'], 0.0)

df_clean[['total_stay', 'total_guests', 'is_room_changed', 'is_family', 'total_cost']].head()

,total_stay,total_guests,is_room_changed,is_family,total_cost
0,0,2,0,0,0.00
1,0,2,0,0,0.00
2,1,1,1,0,75.00
3,1,1,0,0,75.00
4,2,2,0,0,196.00


## 7. Bước 6: Kiểm tra & So sánh dữ liệu Trước vs Sau khi làm sạch

In [10]:
comparison = pd.DataFrame({
'Chỉ số': [
'Tổng số dòng', 
'Tổng số cột', 
'Số dòng trùng lặp', 
'Tổng giá trị Null', 
'Min ADR', 
'Max ADR',
'Tỷ lệ hủy phòng (%)'
],
'Dữ liệu ban đầu (df)': [
f"{df.shape[0]:,}",
df.shape[1],
f"{df.duplicated().sum():,}",
f"{df.isnull().sum().sum():,}",
df['adr'].min(),
df['adr'].max(),
f"{df['is_canceled'].mean()*100:.2f}%"
],
'Dữ liệu sau làm sạch (df_clean)': [
f"{df_clean.shape[0]:,}",
df_clean.shape[1],
f"{df_clean.duplicated().sum():,}",
f"{df_clean.isnull().sum().sum():,}",
df_clean['adr'].min(),
df_clean['adr'].max(),
f"{df_clean['is_canceled'].mean()*100:.2f}%"
]
})

comparison

,Chỉ số,Dữ liệu ban đầu (df),Dữ liệu sau làm sạch (df_clean)
0,Tổng số dòng,"119,390","87,223"
1,Tổng số cột,32,38
2,Số dòng trùng lặp,"31,994",0
3,Tổng giá trị Null,"129,425",0
4,Min ADR,-6.38,0.00
5,Max ADR,5400.00,510.00
6,Tỷ lệ hủy phòng (%),37.04%,27.52%


### Xem 5 dòng đầu tiên của tập dữ liệu sạch (`df_clean`)

In [11]:
df_clean.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date,arrival_date,total_stay,total_guests,is_room_changed,is_family,total_cost
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,0,0,BB,PRT,Direct,Direct,0,0,0,C,C,3,No Deposit,0,0,0,Transient,0.00,0,0,Check-Out,2015-07-01,2015-07-01,0,2,0,0,0.00
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,0,0,BB,PRT,Direct,Direct,0,0,0,C,C,4,No Deposit,0,0,0,Transient,0.00,0,0,Check-Out,2015-07-01,2015-07-01,0,2,0,0,0.00
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,0,0,BB,GBR,Direct,Direct,0,0,0,A,C,0,No Deposit,0,0,0,Transient,75.00,0,0,Check-Out,2015-07-02,2015-07-01,1,1,1,0,75.00
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,0,0,BB,GBR,Corporate,Corporate,0,0,0,A,A,0,No Deposit,304,0,0,Transient,75.00,0,0,Check-Out,2015-07-02,2015-07-01,1,1,0,0,75.00
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,0,0,BB,GBR,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,240,0,0,Transient,98.00,0,1,Check-Out,2015-07-03,2015-07-01,2,2,0,0,196.00


In [12]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 87223 entries, 0 to 119389
Data columns (total 38 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   hotel                           87223 non-null  object        
 1   is_canceled                     87223 non-null  int64         
 2   lead_time                       87223 non-null  int64         
 3   arrival_date_year               87223 non-null  int64         
 4   arrival_date_month              87223 non-null  object        
 5   arrival_date_week_number        87223 non-null  int64         
 6   arrival_date_day_of_month       87223 non-null  int64         
 7   stays_in_weekend_nights         87223 non-null  int64         
 8   stays_in_week_nights            87223 non-null  int64         
 9   adults                          87223 non-null  int64         
 10  children                        87223 non-null  int64         
 11  babies

## 8. Kết luận & Hướng dẫn sử dụng

**Tập dữ liệu đã được làm sạch toàn diện**:
- Đã loại bỏ **31,994** dòng trùng lặp và các đơn **0 khách**.
- Đã điền đầy đủ tất cả các giá trị Missing values (0 missing values còn lại).
- Đã chuẩn hóa giá trị các cột `meal`, `adr` (loại bỏ giá trị âm và ngoại lai 5400).
- Đã chuẩn hóa kiểu dữ liệu cho `children`, `agent`, `company`, `reservation_status_date`, `arrival_date`.
- Đã tạo thêm **5 đặc trưng mới** phục vụ phân tích chuyên sâu (`total_stay`, `total_guests`, `is_room_changed`, `is_family`, `total_cost`).

**Ghi chú**: Theo yêu cầu, dữ liệu sạch hiện được lưu trong biến bộ nhớ **`df_clean`** và chưa được xuất ra file CSV.
*(Nếu bạn muốn lưu file sạch ra đĩa trong tương lai, chỉ cần thực thi dòng lệnh: `df_clean.to_csv('Data/hotel_bookings_cleaned.csv', index=False)`)*.